# Kronos & LightGBM 5-Minute Backtesting Suite

This notebook provides an interactive environment to run, evaluate, and visualize directional backtests on **5-minute BTC and ETH windows** across three benchmarks:

1. **Kronos Foundation Model** (`NeoQuasar/Kronos-small`): Multi-path candlestick autoregressive foundation model predicting $P(\text{up})$.
2. **LightGBM Classifier**: Locally trained tabular gradient boosted tree model using 200-candle lookback features (RSI, Bollinger Bands, volatility, momentum, volume).
3. **Polymarket Benchmark & Momentum Baseline**: Market ask prices from Polymarket CLOB and simple 4-candle momentum.

## 1. Environment Setup & Dependencies

Ensure you have the required packages installed in your Python environment:
```bash
pip install torch lightgbm pandas numpy matplotlib seaborn huggingface_hub einops tqdm
```

In [ ]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
if "." not in sys.path:
    sys.path.insert(0, ".")

import torch
import lightgbm as lgb

# Set device: MPS (Apple Silicon), CUDA, or CPU
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Execution device: {device}")

## 2. Load & Inspect Candle Data

The repository includes cached 5-minute Coinbase candle data under `data/BTC.csv` and `data/ETH.csv`.

In [ ]:
from backtest_big import load_asset, LOOKBACK, STRIDE, cuts_for

btc_df = load_asset("BTC")
eth_df = load_asset("ETH")

print(f"BTC: {len(btc_df):,} candles from {btc_df['timestamps'].iloc[0]} to {btc_df['timestamps'].iloc[-1]} UTC")
print(f"ETH: {len(eth_df):,} candles from {eth_df['timestamps'].iloc[0]} to {eth_df['timestamps'].iloc[-1]} UTC")

btc_cuts = cuts_for(btc_df)
eth_cuts = cuts_for(eth_df)
print(f"Walk-forward test windows (stride={STRIDE}, lookback={LOOKBACK}): BTC={len(btc_cuts)}, ETH={len(eth_cuts)}")

# Plot recent price and volume
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
btc_sample = btc_df.iloc[-500:]
ax1.plot(pd.to_datetime(btc_sample['timestamps']), btc_sample['close'], color='#f7931a', label='BTC Close')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')

ax2.bar(pd.to_datetime(btc_sample['timestamps']), btc_sample['volume'], color='gray', alpha=0.5, label='Volume')
ax2.set_ylabel('Volume')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')
plt.title('BTC 5-Minute Candles (Last 500 Bars)')
plt.tight_layout()
plt.show()

## 3. LightGBM Walk-Forward Backtest

We extract technical features from the 200-candle lookback window and run a walk-forward chronological train/val/test split (60% train, 7-window embargo, remaining test).

In [ ]:
from lightgbm_backtest import features, run as run_lgbm_backtest

print("Running LightGBM walk-forward backtest on BTC & ETH...")
lgbm_btc = run_lgbm_backtest("BTC")
lgbm_eth = run_lgbm_backtest("ETH")

print("\n--- BTC LightGBM Results ---")
print(f"Test Windows: {lgbm_btc['test_n']}")
print(f"Accuracy:     {lgbm_btc['lgbm_test']['accuracy']:.3%}")
print(f"Brier Score:  {lgbm_btc['lgbm_test']['brier']:.4f}")
print(f"Log Loss:     {lgbm_btc['lgbm_test']['logloss']:.4f}")

print("\n--- ETH LightGBM Results ---")
print(f"Test Windows: {lgbm_eth['test_n']}")
print(f"Accuracy:     {lgbm_eth['lgbm_test']['accuracy']:.3%}")
print(f"Brier Score:  {lgbm_eth['lgbm_test']['brier']:.4f}")
print(f"Log Loss:     {lgbm_eth['lgbm_test']['logloss']:.4f}")

In [ ]:
# Feature Importance Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

feats_btc = lgbm_btc['top_features']
ax1.barh([f[0] for f in feats_btc[::-1]], [f[1] for f in feats_btc[::-1]], color='#2b5c8f')
ax1.set_title('BTC Top Feature Importance')
ax1.set_xlabel('Split Count')

feats_eth = lgbm_eth['top_features']
ax2.barh([f[0] for f in feats_eth[::-1]], [f[1] for f in feats_eth[::-1]], color='#5c8f2b')
ax2.set_title('ETH Top Feature Importance')
ax2.set_xlabel('Split Count')

plt.tight_layout()
plt.show()

## 4. Kronos Foundation Model Backtest

Kronos uses multi-path autoregressive sampling (`predict_batch`) to compute $P(\text{up})$.

We can inspect pre-computed results (`results_BTC.json`, `results_ETH.json`) or run the model live.

In [ ]:
# Load pre-computed Kronos results from repository
with open('results_BTC.json') as f:
    res_btc_kronos = json.load(f)
with open('results_ETH.json') as f:
    res_eth_kronos = json.load(f)

print("=== Kronos Foundation Model Backtest ===")
print(f"BTC: Windows={res_btc_kronos['windows']}, Accuracy={res_btc_kronos['kronos']['accuracy']:.3%}, Brier={res_btc_kronos['kronos']['brier']:.4f}")
print(f"ETH: Windows={res_eth_kronos['windows']}, Accuracy={res_eth_kronos['kronos']['accuracy']:.3%}, Brier={res_eth_kronos['kronos']['brier']:.4f}")

In [ ]:
# Optional: Run Kronos inference live on a test batch
run_live_kronos = False  # Set to True to initialize model and run probe inference

if run_live_kronos:
    from model import Kronos, KronosTokenizer, KronosPredictor
    print("Loading Kronos tokenizer and model from Hugging Face...")
    tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
    model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
    predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)
    
    # Test on a small slice
    test_cuts = btc_cuts[:5]
    from backtest_big import kronos_probs
    probs = kronos_probs(predictor, btc_df, test_cuts, tag="probe")
    print("Sample predicted P(up):", probs)

## 5. Model Calibration & Reliability Diagrams

A calibrated model means that when it predicts $P(\text{up}) = 0.70$, the market should go up 70% of the time.
Here we compare the calibration buckets of Kronos vs. the realized win rates.

In [ ]:
calib = res_btc_kronos['calibration']
pred_p = [c['mean_p'] for c in calib]
real_p = [c['realized'] for c in calib]
counts = [c['n'] for c in calib]

plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
plt.scatter(pred_p, real_p, s=[n * 2 for n in counts], color='red', alpha=0.7, edgecolors='black', label='Kronos BTC Buckets (size=N)')
plt.plot(pred_p, real_p, color='red', alpha=0.5)

plt.title('Kronos BTC Reliability / Calibration Curve')
plt.xlabel('Mean Predicted P(up)')
plt.ylabel('Realized P(up)')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left')
plt.show()

## 6. Simulated Trading Performance

We simulate placing a $1 bet when the model's confidence exceeds a given threshold against a flat 50c market (paying a 51c ask, 1c half-spread).

In [ ]:
sim_lgbm = lgbm_btc['lgbm_sim']
sim_kronos = res_btc_kronos['sim_flat_market']

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70]
lgbm_rois = [sim_lgbm[str(th)]['roi_pct'] if str(th) in sim_lgbm and sim_lgbm[str(th)]['bets'] > 0 else 0 for th in thresholds]
kronos_rois = [sim_kronos[str(th)]['roi_pct'] if str(th) in sim_kronos and sim_kronos[str(th)]['bets'] > 0 else 0 for th in thresholds]

x = np.arange(len(thresholds))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, lgbm_rois, width, label='LightGBM ROI %', color='#2b5c8f')
plt.bar(x + width/2, kronos_rois, width, label='Kronos ROI %', color='#c44e52')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.xticks(x, [f">= {th}" for th in thresholds])
plt.xlabel('Confidence Threshold')
plt.ylabel('Simulated ROI (%)')
plt.title('Simulated Trading ROI vs Confidence Threshold (Flat 50c Market, 1c Ask)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Head-to-Head Scorecard Summary

In [ ]:
scorecard_data = {
    "Model": ["LightGBM (BTC)", "LightGBM (ETH)", "Kronos (BTC)", "Kronos (ETH)", "Momentum Baseline (BTC)"],
    "Accuracy": [
        f"{lgbm_btc['lgbm_test']['accuracy']:.1%}",
        f"{lgbm_eth['lgbm_test']['accuracy']:.1%}",
        f"{res_btc_kronos['kronos']['accuracy']:.1%}",
        f"{res_eth_kronos['kronos']['accuracy']:.1%}",
        f"{res_btc_kronos['momentum_accuracy']:.1%}"
    ],
    "Brier Score": [
        f"{lgbm_btc['lgbm_test']['brier']:.4f}",
        f"{lgbm_eth['lgbm_test']['brier']:.4f}",
        f"{res_btc_kronos['kronos']['brier']:.4f}",
        f"{res_eth_kronos['kronos']['brier']:.4f}",
        "N/A"
    ],
    "Log Loss": [
        f"{lgbm_btc['lgbm_test']['logloss']:.4f}",
        f"{lgbm_eth['lgbm_test']['logloss']:.4f}",
        f"{res_btc_kronos['kronos']['logloss']:.4f}",
        f"{res_eth_kronos['kronos']['logloss']:.4f}",
        "N/A"
    ]
}

df_scorecard = pd.DataFrame(scorecard_data)
display(df_scorecard)